### PyTorch CustomNet Exercises

Welcome to the PyTorch CustomNet exercise template notebook.

There are several questions in this notebook and it's your goal to answer them by writing Python and PyTorch code.

> **Note:** There may be more than one solution to each of the exercises, don't worry too much about the *exact* right answer. Try to write some code that works first and then improve it if you can.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import requests
from zipfile import ZipFile
from io import BytesIO

# Load TinyImageSet data
dataset_path = "http://cs231n.stanford.edu/tiny-imagenet-200.zip"

response = requests.get(dataset_path)
# Check if the request was successful
if response.status_code == 200:
    # Open the downloaded bytes and extract them
    with ZipFile(BytesIO(response.content)) as zip_file:
        zip_file.extractall('/dataset')
    print('Download and extraction complete!')
import os
import shutil

val_dir = '/dataset/tiny-imagenet-200/val'
val_annotations_path = os.path.join(val_dir, 'val_annotations.txt')

# Read the annotations file
if os.path.exists(val_annotations_path):
    with open(val_annotations_path, 'r') as f:
        for line in f.readlines():
            parts = line.split('\t')
            img_name, val_class = parts[0], parts[1]

            # Create a subfolder for the class if it doesn't exist
            class_dir = os.path.join(val_dir, val_class)
            os.makedirs(class_dir, exist_ok=True)

            # Move the image from /val/images/ to /val/class_name/
            src = os.path.join(val_dir, 'images', img_name)
            dst = os.path.join(class_dir, img_name)
            if os.path.exists(src):
                shutil.move(src, dst)

    print("Validation directory restructured successfully!")



Download and extraction complete!


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import requests
from zipfile import ZipFile
from io import BytesIO

# Define the custom neural network
class CustomNet(nn.Module):
    def __init__(self):
        super(CustomNet, self).__init__()
        # Define layers of the neural network
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        #self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        #self.conv3 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        # The input image is 224x224.
        # After conv1 (224x224) -> pool (112x112)
        # After conv2 (112x112) -> pool (56x56)
        # After conv3 (56x56) -> pool (28x28)
        # Number of channels after conv3 is 256.
        # So, the flattened size before fc1 is 256 * 28 * 28 = 200704
        self.fc1 = nn.Linear(in_features=64 * 112 * 112, out_features=200) # 200 is the number of classes in TinyImageNet

    def forward(self, x):
        x = self.pool(nn.functional.relu(self.conv1(x)))
        #x = self.pool(nn.functional.relu(self.conv2(x)))
        #x = self.pool(nn.functional.relu(self.conv3(x)))
        x = torch.flatten(x, 1) # Flatten all dimensions except batch
        logits = self.fc1(x)
        return logits

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Define the custom neural network
class CustomNet(nn.Module):
    def __init__(self):
        super(CustomNet, self).__init__()
        # Convolutional layers
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.pool = nn.MaxPool2d(2, 2)

        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(128)

        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(256)

        # Fully connected layers
        # Input image: 224x224
        # After conv1 (64 filters) + pool (2x2): 64 channels, 112x112
        # After conv2 (128 filters) + pool (2x2): 128 channels, 56x56
        # After conv3 (256 filters) + pool (2x2): 256 channels, 28x28
        # Flattened size before fc1: 256 * 28 * 28 = 200704
        self.fc1 = nn.Linear(256 * 28 * 28, 1024)
        self.dropout = nn.Dropout(0.5) # Add dropout for regularization
        self.fc2 = nn.Linear(1024, 200) # 200 classes for TinyImageNet

    def forward(self, x):
        x = self.pool(nn.functional.relu(self.bn1(self.conv1(x))))
        x = self.pool(nn.functional.relu(self.bn2(self.conv2(x))))
        x = self.pool(nn.functional.relu(self.bn3(self.conv3(x))))
        x = torch.flatten(x, 1) # Flatten all dimensions except batch
        x = nn.functional.relu(self.fc1(x))
        x = self.dropout(x)
        logits = self.fc2(x)
        return logits



KeyboardInterrupt: 

In [ ]:
# Define transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize to fit the input dimensions of the network
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Load TinyImageNet
train_dataset = datasets.ImageFolder('/dataset/tiny-imagenet-200/train', transform=transform)
test_dataset = datasets.ImageFolder('/dataset/tiny-imagenet-200/val', transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Initialize the model
model = CustomNet().cuda()

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training loop
def train(epoch, model, train_loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (inputs, targets) in enumerate(train_loader):
        inputs, targets = inputs.cuda(), targets.cuda()
        outputs = model(inputs)
        loss = criterion(outputs, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

    train_loss = running_loss / len(train_loader)
    train_accuracy = 100. * correct / total
    print(f'Train Epoch: {epoch} Loss: {train_loss:.6f} Acc: {train_accuracy:.2f}%')

# Test loop
def test(model, test_loader, criterion):
    model.eval()
    test_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_idx, (inputs, targets) in enumerate(test_loader):
            inputs, targets = inputs.cuda(), targets.cuda()
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            test_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

    test_loss = test_loss / len(test_loader)
    test_accuracy = 100. * correct / total
    print(f'Test Loss: {test_loss:.6f} Acc: {test_accuracy:.2f}%')
    return test_accuracy

# Run the training and testing
num_epochs = 10
for epoch in range(1, num_epochs + 1):
    train(epoch, model, train_loader, criterion, optimizer)
test_accuracy = test(model, test_loader, criterion)

KeyboardInterrupt: 

In [ ]:
import os
import shutil
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import requests
from zipfile import ZipFile
from io import BytesIO

# ---------------------------------------------------------
# 1. DOWNLOAD AND EXTRACT
# ---------------------------------------------------------
dataset_path = "http://cs231n.stanford.edu/tiny-imagenet-200.zip"
response = requests.get(dataset_path)

if response.status_code == 200:
    with ZipFile(BytesIO(response.content)) as zip_file:
        zip_file.extractall('/dataset')
    print('Download and extraction complete!')

# ---------------------------------------------------------
# 2. FIX THE VALIDATION FOLDER (CRITICAL FOR ACCURACY)
# ---------------------------------------------------------
val_dir = '/dataset/tiny-imagenet-200/val'
val_annotations_path = os.path.join(val_dir, 'val_annotations.txt')

if os.path.exists(val_annotations_path):
    print("Reorganizing validation directory...")
    with open(val_annotations_path, 'r') as f:
        for line in f.readlines():
            parts = line.split('\t')
            img_name, val_class = parts[0], parts[1]

            class_dir = os.path.join(val_dir, val_class)
            os.makedirs(class_dir, exist_ok=True)

            src = os.path.join(val_dir, 'images', img_name)
            dst = os.path.join(class_dir, img_name)
            if os.path.exists(src):
                shutil.move(src, dst)
    print("Validation directory fixed!")

# ---------------------------------------------------------
# 3. FAST CUSTOM NETWORK (WITH GLOBAL AVERAGE POOLING)
# ---------------------------------------------------------
class CustomNet(nn.Module):
    def __init__(self):
        super(CustomNet, self).__init__()

        # Wrapped convolutions in Sequential for cleaner code
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        # This takes your 256x28x28 output and averages it down to 256x1x1
        # This is the secret to fast training with large images!
        self.gap = nn.AdaptiveAvgPool2d(1)

        # Because of GAP, the input is now only 256 instead of 200,704
        self.fc1 = nn.Linear(256, 1024)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(1024, 200)

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        x = torch.flatten(x, 1)
        x = nn.functional.relu(self.fc1(x))
        x = self.dropout(x)
        logits = self.fc2(x)
        return logits

# ---------------------------------------------------------
# 4. DATALOADERS & TRAINING LOOP
# ---------------------------------------------------------
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

import os
import shutil

leftover_images_dir = '/dataset/tiny-imagenet-200/val/images'

# Delete the empty 'images' folder if it is still there
if os.path.exists(leftover_images_dir):
    shutil.rmtree(leftover_images_dir)
    print("Cleaned up the empty 'images' directory!")
else:
    print("The directory is already clean.")

train_dataset = datasets.ImageFolder('/dataset/tiny-imagenet-200/train', transform=transform)
test_dataset = datasets.ImageFolder('/dataset/tiny-imagenet-200/val', transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)

model = CustomNet().cuda()

criterion = nn.CrossEntropyLoss()
# Lowered learning rate for stability
optimizer = torch.optim.Adam(model.parameters(), lr=0.0003)

def train(epoch, model, train_loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (inputs, targets) in enumerate(train_loader):
        inputs, targets = inputs.cuda(), targets.cuda()
        outputs = model(inputs)
        loss = criterion(outputs, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

    train_loss = running_loss / len(train_loader)
    train_accuracy = 100. * correct / total
    print(f'Train Epoch: {epoch} Loss: {train_loss:.6f} Acc: {train_accuracy:.2f}%')

def test(model, test_loader, criterion):
    model.eval()
    test_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_idx, (inputs, targets) in enumerate(test_loader):
            inputs, targets = inputs.cuda(), targets.cuda()
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            test_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

    test_loss = test_loss / len(test_loader)
    test_accuracy = 100. * correct / total
    print(f'Test Loss: {test_loss:.6f} Acc: {test_accuracy:.2f}%')
    return test_accuracy

num_epochs = 10
for epoch in range(1, num_epochs + 1):
    train(epoch, model, train_loader, criterion, optimizer)
test_accuracy = test(model, test_loader, criterion)

Download and extraction complete!
Reorganizing validation directory...
Validation directory fixed!
Cleaned up the empty 'images' directory!
Train Epoch: 1 Loss: 4.629592 Acc: 6.11%
Train Epoch: 2 Loss: 4.155845 Acc: 11.39%
Train Epoch: 3 Loss: 3.978872 Acc: 13.70%
Train Epoch: 4 Loss: 3.854208 Acc: 15.63%
Train Epoch: 5 Loss: 3.763461 Acc: 16.98%
Train Epoch: 6 Loss: 3.682744 Acc: 18.21%
Train Epoch: 7 Loss: 3.621814 Acc: 19.26%
Train Epoch: 8 Loss: 3.559870 Acc: 20.30%
Train Epoch: 9 Loss: 3.514497 Acc: 21.04%
Train Epoch: 10 Loss: 3.470416 Acc: 21.94%
Test Loss: 3.430044 Acc: 22.88%
